# 02 기상 영향·회복력 EDA
> 입력: 01의 `preprocessed/` 논문: Thermal comfort and retail sales

### 0-1. 셋업·로드
- 목적: 전처리 데이터 불러오기
- TODO: `preprocessed/` 로드, 경로·폰트
- 확인: 데이터별 shape

In [1]:
# 기본 라이브러리
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 표시 옵션 · 한글 폰트
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 01 노트북의 전처리 결과 폴더
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'dataset').is_dir())
PREP_DIR = ROOT / 'notebooks' / 'preprocessed'
START, END = pd.Timestamp('2025-07-01'), pd.Timestamp('2025-12-31')

# pickle 로드 (01의 6-1에서 저장한 파일명과 동일)
card1 = pd.read_pickle(PREP_DIR / 'card1.pkl')
flow_age = pd.read_pickle(PREP_DIR / 'flow_age.pkl')
flow_time = pd.read_pickle(PREP_DIR / 'flow_time.pkl')
flow_wkdy = pd.read_pickle(PREP_DIR / 'flow_wkdy.pkl')
wx_hourly = pd.read_pickle(PREP_DIR / 'wx_hourly.pkl')
wx_daily = pd.read_pickle(PREP_DIR / 'wx_daily.pkl')
wx_normals = pd.read_pickle(PREP_DIR / 'wx_normals.pkl')
calendar = pd.read_pickle(PREP_DIR / 'calendar.pkl')
industry_map = pd.read_csv(PREP_DIR / 'industry_map.csv', encoding='utf-8-sig')
station_map = pd.read_csv(PREP_DIR / 'station_map.csv', encoding='utf-8-sig')

# 데이터별 shape (데이터2는 부록이라 02에서는 사용하지 않음)
display(pd.DataFrame({
    name: {'행': len(df), '열': df.shape[1]}
    for name, df in [('card1', card1), ('flow_age', flow_age), ('flow_time', flow_time), ('flow_wkdy', flow_wkdy),
                     ('wx_hourly', wx_hourly), ('wx_daily', wx_daily), ('wx_normals', wx_normals),
                     ('calendar', calendar), ('industry_map', industry_map), ('station_map', station_map)]
}).T)

,행,열
card1,1044710,14
flow_age,596252,17
flow_time,570805,29
flow_wkdy,720806,12
wx_hourly,499008,9
wx_daily,22539,12
wx_normals,368,5
calendar,184,12
industry_map,91,6
station_map,4,7


### 0-2. 논문 틀 정리
- 목적: 분석 기준 명시
- TODO: 영향 경로 3가지·논문 vs 우리 데이터 차이표를 마크다운으로 작성
- 확인: 가설 H1~H4, H6~H9a 목록

In [2]:
# 논문: Thermal comfort and retail sales (서울 신한카드 × 고해상도 기온, 준모수 패널 모형)

# (1) 논문 핵심 결과
paper_results = pd.DataFrame([
    ('−15℃ 미만 한파 노출 1일 추가', '매출 +11%', '거래건수 +12%'),
    ('35℃ 초과 폭염 노출 1일 추가', '매출 +4%', '거래건수 +3%'),
    ('−15~−5℃ / 25~35℃ 중간 구간', '매출 −1~2%', '-'),
    ('−5~25℃ 정상 구간', '작거나 유의하지 않음', '업종 이질성 가능성'),
    ('업종', '카페·음식점·편의점·실내시설·연료·의료 증가', '의류 무반응'),
], columns=['조건', '결과', '비고'])

# (2) 날씨 → 매출 경로 3가지와 EDA 신호
channels = pd.DataFrame([
    ('물리적 접근성', '매출 감소 → 날씨 개선 후 반등', '극단일 감소·익일 반등, 야외·이동형 업종 감소'),
    ('온열 쾌적성', '냉난방 공간·상품 매출 증가', '카페·편의점·실내시설·백화점 증가'),
    ('소비 심리', '업종 무관 전반 변동', '업종 간 같은 방향 움직임'),
], columns=['경로', '예측', 'EDA 신호'])

# (3) 논문 vs 우리 데이터 (우리 쪽 수치는 preprocessed 데이터에서 계산)
d25 = wx_daily[(wx_daily['DATE'] >= START) & wx_daily['STATION'].isin(['강남_400', '춘천_101'])]
hot35 = d25.groupby('STATION')['TMAX'].apply(lambda x: int((x >= 35).sum())).to_dict()
min_tmin = d25['TMIN'].min()
n_ind = card1['MCT_RY_CD'].nunique()
comparison = pd.DataFrame([
    ('기간·단위', '2017~2020, 월', f'{START:%Y-%m}~{END:%Y-%m}, 일×6시간', '일·시간대 단위 분석'),
    ('공간', '서울 약 68,000 구역', '강남구·춘천시 (시군구)', '지역 비교(H3)로 활용'),
    ('업종', '63개', f'{n_ind}개', '전체 업종 + 논문 5그룹 대조(H2)'),
    ('기온 자료', '1km 격자, 사인 보간 15분', '관측소 시간자료 실측', '6시간 구간 노출시간 직접 계산'),
    ('폭염 35℃', '분석 핵심', f"강남_400 {hot35.get('강남_400')}일 · 춘천_101 {hot35.get('춘천_101')}일", '표본 작음 → 33℃ 병행'),
    ('한파 −15℃', '분석 핵심', f'최저 {min_tmin:.1f}℃ (해당 없음)', '한파 효과 검증 불가'),
    ('교란', '다년도로 분리', '계절 하강·추석·연말이 기온과 겹침', '월×요일×공휴일 기준선 제거'),
    ('목적', '평균 효과', '취약 상권', '감소·회복 지연 업종에 초점'),
], columns=['항목', '논문', '우리 데이터', 'EDA 대응'])

# (4) 이 노트북의 가설 (H5·H9b 제외)
hypotheses = pd.DataFrame([
    ('H1', '비선형', '기온 구간별 매출 편차가 논문처럼 극단에서 달라지는가', '3'),
    ('H2', '업종 이질성', '전체 업종·논문 5그룹의 반응 방향은', '4'),
    ('H3', '지역', '춘천에서도 온열 쾌적성 효과가 나타나는가 (논문 한계 3)', '5'),
    ('H4', '시간대', '폭염일 낮 소비가 저녁으로 이동하는가', '6'),
    ('H6', '매출 분해', '방문(건수) 감소인가 객단가 변화인가', '7'),
    ('H7', '강수', '호우일 감소와 익일 반등이 있는가', '8'),
    ('H8', '적응', '폭염이 반복될수록 반응이 약해지는가 (논문 한계 1)', '9'),
    ('H9a', '회복력', '극단 기상 후 업종별로 며칠 만에 회복하는가', '10'),
], columns=['ID', '축', '질문', '섹션'])

for title, df in [('논문 핵심 결과', paper_results), ('영향 경로', channels),
                  ('논문 vs 우리 데이터', comparison), ('가설', hypotheses)]:
    print(f'■ {title}')
    display(df)

■ 논문 핵심 결과


,조건,결과,비고
0,−15℃ 미만 한파 노출 1일 추가,매출 +11%,거래건수 +12%
1,35℃ 초과 폭염 노출 1일 추가,매출 +4%,거래건수 +3%
2,−15~−5℃ / 25~35℃ 중간 구간,매출 −1~2%,-
3,−5~25℃ 정상 구간,작거나 유의하지 않음,업종 이질성 가능성
4,업종,카페·음식점·편의점·실내시설·연료·의료 증가,의류 무반응


■ 영향 경로


,경로,예측,EDA 신호
0,물리적 접근성,매출 감소 → 날씨 개선 후 반등,"극단일 감소·익일 반등, 야외·이동형 업종 감소"
1,온열 쾌적성,냉난방 공간·상품 매출 증가,카페·편의점·실내시설·백화점 증가
2,소비 심리,업종 무관 전반 변동,업종 간 같은 방향 움직임


■ 논문 vs 우리 데이터


,항목,논문,우리 데이터,EDA 대응
0,기간·단위,"2017~2020, 월","2025-07~2025-12, 일×6시간",일·시간대 단위 분석
1,공간,"서울 약 68,000 구역",강남구·춘천시 (시군구),지역 비교(H3)로 활용
2,업종,63개,91개,전체 업종 + 논문 5그룹 대조(H2)
3,기온 자료,"1km 격자, 사인 보간 15분",관측소 시간자료 실측,6시간 구간 노출시간 직접 계산
4,폭염 35℃,분석 핵심,강남_400 15일 · 춘천_101 11일,표본 작음 → 33℃ 병행
5,한파 −15℃,분석 핵심,최저 -13.4℃ (해당 없음),한파 효과 검증 불가
6,교란,다년도로 분리,계절 하강·추석·연말이 기온과 겹침,월×요일×공휴일 기준선 제거
7,목적,평균 효과,취약 상권,감소·회복 지연 업종에 초점


■ 가설


,ID,축,질문,섹션
0,H1,비선형,기온 구간별 매출 편차가 논문처럼 극단에서 달라지는가,3
1,H2,업종 이질성,전체 업종·논문 5그룹의 반응 방향은,4
2,H3,지역,춘천에서도 온열 쾌적성 효과가 나타나는가 (논문 한계 3),5
3,H4,시간대,폭염일 낮 소비가 저녁으로 이동하는가,6
4,H6,매출 분해,방문(건수) 감소인가 객단가 변화인가,7
5,H7,강수,호우일 감소와 익일 반등이 있는가,8
6,H8,적응,폭염이 반복될수록 반응이 약해지는가 (논문 한계 1),9
7,H9a,회복력,극단 기상 후 업종별로 며칠 만에 회복하는가,10


## 1. 분석 테이블

### 1-1. 카드 분석 테이블
- 목적: 분석 단위 집계
- TODO: 날짜×6h×지역×업종 매출·건수·단가 (법인·보류 플래그 반영)
- 확인: shape, 빈 셀 비율

### 1-2. 노출 변수
- 목적: 극한 기온 노출 시간 산출
- TODO: 시간자료 → 6h·일 단위 최고기온, 33/35℃ 이상 시간 수, 체감온도, 강수
- 확인: 노출 변수 분포

### 1-3. 결합
- 목적: 카드·기상·달력 병합
- TODO: 지역→지점 매핑 후 병합
- 확인: 병합 누락 건수

## 2. 기준선

### 2-1. 기준선 제거
- 목적: 계절·달력 교란 제거
- TODO: 업종·지역별 월×요일×공휴일 평균 → 로그 편차
- 확인: 편차 분포, 대조군 편차≈0 여부

## 3. H1 비선형

### 3-1. 기온 구간별 편차
- 목적: 비선형 반응 확인
- TODO: 기온 구간별 편차 평균·CI (소매 범위 전체)
- 확인: 반응 곡선 모양

### 3-2. 구간 폭 민감도
- 목적: 구간 선택 영향 점검
- TODO: 구간 폭 2·3·5℃ 비교
- 확인: 곡선 모양 유지 여부

## 4. H2 업종 이질성

### 4-1. 전체 업종 히트맵
- 목적: 업종별 반응 방향
- TODO: 업종×기온구간 편차 히트맵 (등급 A/B)
- 확인: 증가·감소 업종 목록

### 4-2. 폭염 민감도 순위
- 목적: 업종 순위와 불확실성
- TODO: 폭염일 편차 순위, 날짜 부트스트랩 CI
- 확인: 유의한 감소·증가 업종

### 4-3. 논문 5그룹 대조
- 목적: 논문 결과 재현 여부
- TODO: 5그룹 편차를 논문 방향과 비교 표
- 확인: 일치·불일치 그룹

### 4-4. 대조군 점검
- 목적: 달력 교란 잔존 확인
- TODO: 세금공과금 등 대조군 편차 확인
- 확인: 교란 존재 여부

### 4-5. 반응 군집
- 목적: 비슷한 반응 업종 묶기
- TODO: 업종 반응 프로파일 군집화
- 확인: 군집 구성, 논문 그룹과 겹침

## 5. H3 지역

### 5-1. 강남 vs 춘천
- 목적: 지역 차이 (논문 한계 3)
- TODO: 동일 틀로 곡선·업종 순위 비교
- 확인: 지역 간 방향 차이

## 6. H4 시간대

### 6-1. 시간대 이동
- 목적: 폭염 시 소비 시간 변화
- TODO: 시간대×폭염여부 편차 히트맵, 낮→저녁 비중 변화
- 확인: 시간 이동 여부

## 7. H6 매출액 vs 건수·단가

### 7-1. 매출 분해
- 목적: 방문 감소 vs 객단가 변화
- TODO: 매출 편차 = 건수 편차 + 단가 편차
- 확인: 업종별 분해 결과

## 8. H7 강수

### 8-1. 호우 반응
- 목적: 강수 영향 확인
- TODO: 호우일(30·80mm) 선정 → 당일·시간대 편차
- 확인: 강수 민감 업종

## 9. H8 적응

### 9-1. 폭염 순번별 반응
- 목적: 적응 여부 (논문 한계 1)
- TODO: 폭염 기간 순번별 편차 비교
- 확인: 반응 약화 여부

## 10. H9a 회복력

### 10-1. 이벤트 정의
- 목적: 회복력 분석 대상 확정
- TODO: 폭염 기간·호우일 이벤트화, 창 겹침 표시
- 확인: 이벤트 목록·수

### 10-2. 이벤트 스터디
- 목적: 이벤트 전후 추이
- TODO: -7~+14일 업종별 편차 추이
- 확인: 업종별 추이 그래프

### 10-3. 회복 지표
- 목적: 업종별 회복력 수치화
- TODO: 최대 낙폭·회복일수·누적손실·반등 산출
- 확인: 업종별 회복력 표

### 10-4. 당일 회복
- 목적: 같은 날 내 회복 여부
- TODO: 6h 단위 이벤트일 시간대별 편차
- 확인: 시간대별 회복 패턴

## 11. 유동인구 보조

### 11-1. 여름 유동 구조
- 목적: 유동인구로 H3·H4 보조
- TODO: 8월 vs 9월 시간대·요일 구조, 월 변동 큰 격자
- 확인: 여름 유동 감소 격자·시간대

## 12. 종합

### 12-1. 취약 후보 표
- 목적: 결과 통합
- TODO: 업종×지역×시간대 저항·회복 지표 통합 순위
- 확인: 취약 후보 목록

### 12-2. 논문 대조·한계
- 목적: 결론 정리
- TODO: 같은 점·다른 점, 한계, 다음 단계 정리
- 확인: 요약 표